In [ ]:
!pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 6.2 MB/s eta 0:00:00


In [ ]:
!pip install deep-translator

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 2.2 MB/s eta 0:00:00


In [ ]:
!pip install transformers sentencepiece

In [ ]:
from docx import Document
from docx.table import _Cell
from typing import List
import re

# ================= SETTINGS =================
INPUT_PATH  = "Sayfiyev Sardorbek Dissertatsiya.docx"
OUTPUT_PATH = "3.1_uz.docx"

TARGET_LANG = "uz"   # "uz" | "ru" | "en"
ENGINE = "google"    # "google" (online) | "marian" (offline)
# ============================================


# --------- Language Detection ---------
def detect_language(text: str) -> str:
    cyrillic = len(re.findall(r"[А-Яа-яЁё]", text))
    latin = len(re.findall(r"[A-Za-z]", text))
    uz_specific = len(re.findall(r"[ʻʼ‘’]", text))

    if cyrillic > latin:
        return "ru"
    if uz_specific > 0:
        return "uz"
    return "en"


# --------- Translator Base ----------
class BaseTranslator:
    def translate(self, text: str) -> str:
        raise NotImplementedError


class GoogleTranslatorAuto(BaseTranslator):
    def __init__(self, source, target):
        from deep_translator import GoogleTranslator
        self._t = GoogleTranslator(source=source, target=target)

    def translate(self, text: str) -> str:
        if not text.strip():
            return text
        return self._t.translate(text)


class MarianTranslatorAuto(BaseTranslator):
    def __init__(self, source, target):
        from transformers import MarianMTModel, MarianTokenizer
        model_name = f"Helsinki-NLP/opus-mt-{source}-{target}"
        self.tokenizer = MarianTokenizer.from_pretrained(model_name)
        self.model = MarianMTModel.from_pretrained(model_name)

    def translate(self, text: str) -> str:
        if not text.strip():
            return text
        batch = self.tokenizer(text, return_tensors="pt", truncation=True)
        gen = self.model.generate(**batch, max_new_tokens=512)
        return self.tokenizer.decode(gen[0], skip_special_tokens=True)


def get_translator(source, target):
    if ENGINE == "google":
        return GoogleTranslatorAuto(source, target)
    elif ENGINE == "marian":
        return MarianTranslatorAuto(source, target)
    else:
        raise ValueError("ENGINE must be 'google' or 'marian'")


# --------- DOCX Helpers ----------
def write_text_to_runs(paragraph, new_text: str):
    runs = paragraph.runs
    if not runs:
        paragraph.add_run(new_text)
        return

    lengths = [len(r.text) for r in runs]
    pos = 0

    for i, r in enumerate(runs):
        take = lengths[i]
        # Ensure new_text is treated as a string, even if it's None or very short.
        # The original text was already handled as a string, so this should be fine.
        r.text = str(new_text)[pos:pos+take]
        pos += take

    if pos < len(str(new_text)):
        runs[-1].text += str(new_text)[pos:]
    # If new_text is shorter than original, clear remaining runs (if any)
    elif pos > len(str(new_text)):
        # This part might not be strictly necessary if new_text is handled correctly as str
        # but it's good practice to ensure no leftover text from original runs.
        for r_idx in range(i + 1, len(runs)):
            runs[r_idx].text = ""

def iter_paragraphs_in_cell(cell: _Cell):
    for p in cell.paragraphs:
        yield p
    for tbl in cell.tables:
        for row in tbl.rows:
            for c in row.cells:
                yield from iter_paragraphs_in_cell(c)


def translate_paragraph(paragraph, translator):
    original = paragraph.text
    if not original.strip():
        return
    translated = translator.translate(original)
    # Handle cases where translation might return None (e.g., API issues)
    if translated is None:
        print(f"Warning: Translation returned None for text: '{original}'. Keeping original text.")
        translated = original # Keep original text if translation fails
    if translated != original:
        write_text_to_runs(paragraph, translated)


def translate_document(doc: Document, translator):
    for p in doc.paragraphs:
        translate_paragraph(p, translator)

    for tbl in doc.tables:
        for row in tbl.rows:
            for cell in row.cells:
                for p in iter_paragraphs_in_cell(cell):
                    translate_paragraph(p, translator)

    for section in doc.sections:
        for p in section.header.paragraphs:
            translate_paragraph(p, translator)
        for p in section.footer.paragraphs:
            translate_paragraph(p, translator)


# --------- MAIN ----------
def main():
    print("Loading document...")
    doc = Document(INPUT_PATH)

    # First non-empty paragraph for detection
    sample_text = ""
    for p in doc.paragraphs:
        if p.text.strip():
            sample_text = p.text
            break

    source_lang = detect_language(sample_text)

    if source_lang == TARGET_LANG:
        print("Source and target languages are the same. Nothing to translate.")
        return

    print(f"Detected source language: {source_lang}")
    print(f"Target language: {TARGET_LANG}")
    print(f"Engine: {ENGINE}")

    translator = get_translator(source_lang, TARGET_LANG)
    translate_document(doc, translator)

    doc.save(OUTPUT_PATH)
    print(f"Saved: {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

Loading document...
Source and target languages are the same. Nothing to translate.
